# CloudCompute: ensemble weight search
Поиск весов по совместимым OOF validation-предсказаниям.

In [ ]:
REPO_DIR = "/root/text-orientation-classification"
STATE_DIR = "/root/text-orientation-state"
CANDIDATES = {
    "mobilenet_v3_large_robust": f"{STATE_DIR}/training/runs/robust/mobilenet_v3_large/full",
    "vit_b_16": f"{STATE_DIR}/training/runs/vit_b_16/full",
}
WEIGHT_STEP = 0.05
PROMOTE_ENSEMBLE = True
RUN_TESTS = False

In [ ]:
import os, subprocess, sys
from pathlib import Path
os.chdir(REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
if RUN_TESTS: subprocess.run([sys.executable, "-m", "pytest", "-q"], check=True)
for selector, path in CANDIDATES.items():
    missing = [name for name in ("best.pt", "calibration.json", "validation_predictions.npz") if not (Path(path) / name).is_file()]
    if missing: raise FileNotFoundError(f"{selector}: missing {missing} in {path}")

In [ ]:
command = [sys.executable, "-m", "scripts.search_ensemble", "--project-dir", STATE_DIR, "--weight-step", str(WEIGHT_STEP)]
for selector, path in CANDIDATES.items(): command += ["--candidate", f"{selector}={path}"]
if not PROMOTE_ENSEMBLE: command.append("--no-promote")
subprocess.run(command, check=True)
print("Report:", Path(STATE_DIR) / "evaluation/ensembles/search.json")